In [1]:
import numpy as np
import pandas as pd
from sklearn.decomposition import NMF
from sklearn.preprocessing import MinMaxScaler
# from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

In [2]:
df = pd.read_csv(r'pipeline_cache\ampds_behavior_context_labeled_features.csv')
print(df.head())

                   window_id  CWE__last_location_of_minimum  \
0  2012-09-01 07:00:00+00:00                            1.0   
1  2012-09-01 07:15:00+00:00                            1.0   
2  2012-09-01 07:30:00+00:00                            1.0   
3  2012-09-01 07:45:00+00:00                            1.0   
4  2012-09-01 08:00:00+00:00                            1.0   

   CWE__first_location_of_maximum  CWE__number_crossing_m__m_0  \
0                             0.0                          0.0   
1                             0.0                          0.0   
2                             0.0                          0.0   
3                             0.0                          0.0   
4                             0.0                          0.0   

   CWE__number_crossing_m__m_1  HPE__symmetry_looking__r_0.8500000000000001  \
0                          0.0                                          1.0   
1                          0.0                                    

In [24]:
df.columns

Index(['window_id', 'CWE__last_location_of_minimum',
       'CWE__first_location_of_maximum', 'CWE__number_crossing_m__m_0',
       'CWE__number_crossing_m__m_1',
       'HPE__symmetry_looking__r_0.8500000000000001',
       'HPE__symmetry_looking__r_0.7000000000000001',
       'HPE__symmetry_looking__r_0.75', 'HPE__symmetry_looking__r_0.8',
       'HPE__symmetry_looking__r_0.55',
       ...
       'wx_Cloudy', 'wx_Fog', 'wx_Rain', 'hour_sin', 'hour_cos', 'dow_sin',
       'dow_cos', 'is_weekend', 'is_anomaly', 'anomaly_type'],
      dtype='str', length=833)

In [3]:
dropping_cols = ["window_id", "anomaly_type"]
label = "is_anomaly"
df = df.drop(columns=dropping_cols)
df_copy = df.copy()
X = df_copy.drop(columns=label)
y = df[label]

print(X.shape)
print(y.shape)

(5856, 830)
(5856,)


In [4]:
n = len(X)

train_end = int(0.70 * n)
val_end = int(0.85 * n)

X_train = X.iloc[:train_end]
y_train = y.iloc[:train_end]

X_val = X.iloc[train_end:val_end]
y_val = y.iloc[train_end:val_end]

X_test = X.iloc[val_end:]
y_test = y.iloc[val_end:]

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(4099, 830) (4099,)
(878, 830) (878,)
(879, 830) (879,)


In [5]:
# Training the NMF on training normal data
y_train_0 = y_train[y_train == 0]
y_train_1 = y_train[y_train == 1]

y_test_0 = y_test[y_test == 0]
y_test_1 = y_test[y_test == 1]

print(len(y_test_0))
print(len(y_test_1))

# print(len(y_train_0))
# print(len(y_train_1))

X_train_normal = X_train[y_train == 0]
# print(len(X_train_normal))


# Scaling to 0 and 1
scaler = MinMaxScaler()

# X_train_normal_scaled = scaler.fit_transform(X_train_normal)
# # X_val_scaled = scaler.transform(X_val)
# # X_test_scaled = scaler.transform(X_test)

X_train_normal_scaled = scaler.fit_transform(X_train_normal)
X_val_scaled = np.clip(scaler.transform(X_val), 0, None)
X_test_scaled = np.clip(scaler.transform(X_test), 0, None)



786
93


In [10]:
def reconstruction_error_per_sample(X_ori, X_recon):
    return np.linalg.norm(X_ori - X_recon, axis=1)

def fit_and_score_nmf(
    X_train_normal_scaled,
    X_val_scaled,
    y_val,
    n_components,
    random_state=42,
):
    start = time.perf_counter()

    nmf = NMF(
        n_components=n_components,
        init="nndsvda",
        solver="cd",
        max_iter=1000,
        random_state=random_state,
    )

    print(f"\n{'='*60}")
    print(f"Training NMF with n_components = {n_components}")

    nmf.fit(X_train_normal_scaled)

    fit_time = time.perf_counter() - start

    print(f"Finished fitting in {fit_time:.2f} seconds")
    print(f"Iterations used: {nmf.n_iter_}/{nmf.max_iter}")

    if nmf.n_iter_ == nmf.max_iter:
        print("Status: Reached maximum iterations (did NOT fully converge)")
    else:
        print("Status: Converged")

    # Validation
    W_val = nmf.transform(X_val_scaled)
    X_val_recon = nmf.inverse_transform(W_val)
    val_scores = reconstruction_error_per_sample(X_val_scaled, X_val_recon)

    roc_auc = roc_auc_score(y_val, val_scores)
    pr_auc = average_precision_score(y_val, val_scores)

    thresholds = np.unique(
        np.quantile(val_scores, np.linspace(0.01, 0.99, 99))
    )

    best = {
        "threshold": None,
        "f1": -1,
        "precision": None,
        "recall": None,
    }

    for t in thresholds:
        preds = (val_scores >= t).astype(int)

        p = precision_score(y_val, preds, zero_division=0)
        r = recall_score(y_val, preds, zero_division=0)
        f1 = f1_score(y_val, preds, zero_division=0)

        if f1 > best["f1"]:
            best = {
                "threshold": float(t),
                "f1": float(f1),
                "precision": float(p),
                "recall": float(r),
            }

    print(
        f"ROC-AUC={roc_auc:.4f} | PR-AUC={pr_auc:.4f} | "
        f"Best F1={best['f1']:.4f}"
    )

    return {
        "model": nmf,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "best_threshold": best["threshold"],
        "best_val_f1": best["f1"],
        "best_val_precision": best["precision"],
        "best_val_recall": best["recall"],
        "val_scores": val_scores,
        "fit_time": fit_time,
        "n_iter": nmf.n_iter_,
    }

In [12]:
import time
n_components = [80, 120, 150, 180, 200]

results = []
models = {}

overall_start = time.perf_counter()

for i, n_comps in enumerate(n_components, start=1):

    print(f"\n[{i}/{len(n_components)}] Starting model with {n_comps} components...")

    out = fit_and_score_nmf(
        X_train_normal_scaled,
        X_val_scaled,
        y_val,
        n_components=n_comps,
    )

    results.append({
        "n_components": n_comps,
        "iterations": out["n_iter"],
        "fit_time_sec": round(out["fit_time"], 2),
        "roc_auc": out["roc_auc"],
        "pr_auc": out["pr_auc"],
        "best_val_f1": out["best_val_f1"],
        "best_val_precision": out["best_val_precision"],
        "best_val_recall": out["best_val_recall"],
        "best_threshold": out["best_threshold"],
    })

    models[n_comps] = out

overall_time = time.perf_counter() - overall_start

results_df = (
    pd.DataFrame(results)
    .sort_values(["pr_auc", "roc_auc"], ascending=False)
)

print("\n" + "="*70)
print("Final Results")
print("="*70)
print(results_df.to_string(index=False))
print(f"\nTotal runtime: {overall_time:.2f} seconds")


[1/5] Starting model with 80 components...

Training NMF with n_components = 80


c:\1.Revanth\Projects\research\.venv\Lib\site-packages\sklearn\decomposition\_nmf.py:1723: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


Finished fitting in 18.44 seconds
Iterations used: 1000/1000
Status: Reached maximum iterations (did NOT fully converge)
ROC-AUC=0.7307 | PR-AUC=0.3498 | Best F1=0.4368

[2/5] Starting model with 120 components...

Training NMF with n_components = 120


c:\1.Revanth\Projects\research\.venv\Lib\site-packages\sklearn\decomposition\_nmf.py:1723: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


Finished fitting in 62.69 seconds
Iterations used: 1000/1000
Status: Reached maximum iterations (did NOT fully converge)
ROC-AUC=0.7418 | PR-AUC=0.3412 | Best F1=0.4505

[3/5] Starting model with 150 components...

Training NMF with n_components = 150


...

Finished fitting in 117.50 seconds
Iterations used: 1000/1000
Status: Reached maximum iterations (did NOT fully converge)
ROC-AUC=0.7545 | PR-AUC=0.3418 | Best F1=0.4396

[4/5] Starting model with 180 components...

Training NMF with n_components = 180


c:\1.Revanth\Projects\research\.venv\Lib\site-packages\sklearn\decomposition\_nmf.py:1723: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


Finished fitting in 422.30 seconds
Iterations used: 1000/1000
Status: Reached maximum iterations (did NOT fully converge)
ROC-AUC=0.7431 | PR-AUC=0.3277 | Best F1=0.4306

[5/5] Starting model with 200 components...

Training NMF with n_components = 200


c:\1.Revanth\Projects\research\.venv\Lib\site-packages\sklearn\decomposition\_nmf.py:1723: ConvergenceWarning: Maximum number of iterations 1000 reached. Increase it to improve convergence.
  warnings.warn(


Finished fitting in 720.10 seconds
Iterations used: 1000/1000
Status: Reached maximum iterations (did NOT fully converge)
ROC-AUC=0.7532 | PR-AUC=0.3279 | Best F1=0.4400

Final Results
 n_components  iterations  fit_time_sec  roc_auc   pr_auc  best_val_f1  best_val_precision  best_val_recall  best_threshold
           80        1000         18.44 0.730723 0.349821     0.436782            0.535211         0.368932        2.218365
          150        1000        117.50 0.754513 0.341754     0.439560            0.506329         0.388350        1.396424
          120        1000         62.69 0.741760 0.341168     0.450549            0.518987         0.398058        1.579394
          200        1000        720.10 0.753223 0.327917     0.440000            0.453608         0.427184        0.855626
          180        1000        422.30 0.743088 0.327651     0.430622            0.424528         0.436893        0.918301

Total runtime: 1442.34 seconds


In [13]:
# Pick best model by PR AUC
best_k = int(results_df.iloc[0]["n_components"])
best_model = models[best_k]["model"]
best_threshold = models[best_k]["best_threshold"]

print(f"\nBest n_components = {best_k}")
print(f"Best validation threshold = {best_threshold:.6f}")


Best n_components = 80
Best validation threshold = 2.218365


In [14]:
# Final Evals
W_test = best_model.transform(X_test_scaled)
X_test_recon = best_model.inverse_transform(W_test)
test_scores = reconstruction_error_per_sample(X_test_scaled, X_test_recon)   

test_pred = (test_scores >= best_threshold).astype(int)
test_roc_auc = roc_auc_score(y_test, test_scores)
test_pr_auc = average_precision_score(y_test, test_scores)
test_precision = precision_score(y_test, test_pred, zero_division=0)
test_recall = recall_score(y_test, test_pred, zero_division=0)
test_f1 = f1_score(y_test, test_pred, zero_division=0)

print(f"ROC AUC: {test_roc_auc:.4f}  PR AUC: {test_pr_auc:.4f}  Precision: {test_precision:.4f}  Recall: {test_recall:.4f}  F1: {test_f1:.4f}")

ROC AUC: 0.6942  PR AUC: 0.3071  Precision: 0.3936  Recall: 0.3978  F1: 0.3957


In [ ]:
# from sklearn.model_selection import train_test_split

# df_reset = df.reset_index(drop=True)

# # split anomalies first, stratified by type, so every scenario appears in every split
# anomaly_df = df_reset[df_reset['is_anomaly'] == 1]
# normal_df = df_reset[df_reset['is_anomaly'] == 0]

# anom_train, anom_temp = train_test_split(
#     anomaly_df, test_size=0.30, stratify=anomaly_df['anomaly_type'], random_state=42
# )
# anom_val, anom_test = train_test_split(
#     anom_temp, test_size=0.50, stratify=anom_temp['anomaly_type'], random_state=42
# )

# norm_train, norm_temp = train_test_split(normal_df, test_size=0.30, random_state=42)
# norm_val, norm_test = train_test_split(norm_temp, test_size=0.50, random_state=42)

# train_df = pd.concat([anom_train, norm_train]).sample(frac=1, random_state=42)
# val_df   = pd.concat([anom_val, norm_val]).sample(frac=1, random_state=42)
# test_df  = pd.concat([anom_test, norm_test]).sample(frac=1, random_state=42)

# print(pd.crosstab(pd.concat([anom_train,anom_val,anom_test])['anomaly_type'],
#                    pd.concat([anom_train.assign(split='train'), anom_val.assign(split='val'), anom_test.assign(split='test')])['split']))

In [16]:
# import numpy as np
# import pandas as pd
# import warnings
# from sklearn.preprocessing import MinMaxScaler
# from sklearn.decomposition import NMF
# from sklearn.metrics import (
#     roc_auc_score, average_precision_score,
#     precision_score, recall_score, f1_score
# )

# # ============================================================
# # REBUILD X/y FROM STRATIFIED SPLITS
# # ============================================================
# drop_cols = ['is_anomaly', 'anomaly_type']

# X_train = train_df.drop(columns=drop_cols)
# y_train = train_df['is_anomaly']

# X_val = val_df.drop(columns=drop_cols)
# y_val = val_df['is_anomaly']

# X_test = test_df.drop(columns=drop_cols)
# y_test = test_df['is_anomaly']

# print(X_train.shape, y_train.shape)
# print(X_val.shape, y_val.shape)
# print(X_test.shape, y_test.shape)

# print("\nTrain anomaly type counts:\n", train_df['anomaly_type'].value_counts())
# print("\nVal anomaly type counts:\n", val_df['anomaly_type'].value_counts())
# print("\nTest anomaly type counts:\n", test_df['anomaly_type'].value_counts())

# # ============================================================
# # TRAIN NMF ON NORMAL DATA ONLY (novelty detection framing)
# # ============================================================
# X_train_normal = X_train[y_train == 0]
# print(f"\nNormal training rows: {len(X_train_normal)}")

# scaler = MinMaxScaler()
# X_train_normal_scaled = scaler.fit_transform(X_train_normal)
# X_val_scaled = np.clip(scaler.transform(X_val), 0, None)
# X_test_scaled = np.clip(scaler.transform(X_test), 0, None)

# neg_val = (scaler.transform(X_val) < 0).sum()
# neg_test = (scaler.transform(X_test) < 0).sum()
# print(f"Clipped negative values — val: {neg_val}, test: {neg_test}")

# # ============================================================
# # NMF FIT + SCORE HELPER
# # ============================================================
# def reconstruction_error_per_sample(X_ori, X_recon):
#     return np.linalg.norm(X_ori - X_recon, axis=1)

# def fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components,
#                        max_iter=1000, tol=1e-3, random_state=42):
#     with warnings.catch_warnings(record=True) as w:
#         warnings.simplefilter("always")
#         nmf = NMF(
#             n_components=n_components,
#             init="nndsvda",
#             solver="cd",
#             max_iter=max_iter,
#             tol=tol,
#             random_state=random_state,
#         )
#         nmf.fit(X_train_normal_scaled)
#         converged = len(w) == 0

#     # training reconstruction error — sanity check against memorization at high K
#     train_recon = nmf.inverse_transform(nmf.transform(X_train_normal_scaled))
#     train_err = reconstruction_error_per_sample(X_train_normal_scaled, train_recon).mean()

#     W_val = nmf.transform(X_val_scaled)
#     X_val_recon = nmf.inverse_transform(W_val)
#     val_scores = reconstruction_error_per_sample(X_val_scaled, X_val_recon)

#     roc_auc = roc_auc_score(y_val, val_scores)
#     pr_auc = average_precision_score(y_val, val_scores)

#     thresholds = np.unique(np.quantile(val_scores, np.linspace(0.01, 0.99, 99)))
#     best = {"threshold": None, "f1": -1, "precision": None, "recall": None}
#     for t in thresholds:
#         preds = (val_scores >= t).astype(int)
#         p = precision_score(y_val, preds, zero_division=0)
#         r = recall_score(y_val, preds, zero_division=0)
#         f1 = f1_score(y_val, preds, zero_division=0)
#         if f1 > best["f1"]:
#             best = {"threshold": float(t), "f1": float(f1), "precision": float(p), "recall": float(r)}

#     return {
#         "model": nmf,
#         "converged": converged,
#         "n_iter": nmf.n_iter_,
#         "train_recon_err": train_err,
#         "roc_auc": roc_auc,
#         "pr_auc": pr_auc,
#         "best_threshold": best["threshold"],
#         "best_val_f1": best["f1"],
#         "best_val_precision": best["precision"],
#         "best_val_recall": best["recall"],
#         "val_scores": val_scores,
#     }

# # ============================================================
# # SWEEP N_COMPONENTS
# # ============================================================
# n_components_list = [60, 80, 100, 120, 150]

# results = []
# models = {}

# for n_comps in n_components_list:
#     out = fit_and_score_nmf(X_train_normal_scaled, X_val_scaled, y_val, n_components=n_comps)
#     status = "OK" if out["converged"] else "NOT CONVERGED"
#     print(f"K={n_comps:>4} | {status} | n_iter={out['n_iter']:>5} | "
#           f"train_recon_err={out['train_recon_err']:.4f} | "
#           f"roc_auc={out['roc_auc']:.4f} | pr_auc={out['pr_auc']:.4f}")

#     results.append({
#         "n_components": n_comps,
#         "converged": out["converged"],
#         "n_iter": out["n_iter"],
#         "train_recon_err": out["train_recon_err"],
#         "roc_auc": out["roc_auc"],
#         "pr_auc": out["pr_auc"],
#         "best_val_f1": out["best_val_f1"],
#         "best_val_precision": out["best_val_precision"],
#         "best_val_recall": out["best_val_recall"],
#         "best_threshold": out["best_threshold"],
#     })
#     models[n_comps] = out

# results_df = pd.DataFrame(results).sort_values(["pr_auc", "roc_auc"], ascending=False)
# print("\nValidation results:")
# print(results_df.to_string(index=False))

# # ============================================================
# # PICK BEST MODEL BY PR-AUC
# # ============================================================
# best_k = int(results_df.iloc[0]["n_components"])
# best_model = models[best_k]["model"]
# best_threshold = models[best_k]["best_threshold"]

# print(f"\nBest n_components = {best_k}")
# print(f"Best validation threshold = {best_threshold:.6f}")
# print(f"Converged: {models[best_k]['converged']} (n_iter={models[best_k]['n_iter']})")

# # ============================================================
# # FINAL TEST EVALUATION
# # ============================================================
# W_test = best_model.transform(X_test_scaled)
# X_test_recon = best_model.inverse_transform(W_test)
# test_scores = reconstruction_error_per_sample(X_test_scaled, X_test_recon)

# test_pred = (test_scores >= best_threshold).astype(int)

# test_roc_auc = roc_auc_score(y_test, test_scores)
# test_pr_auc = average_precision_score(y_test, test_scores)
# test_precision = precision_score(y_test, test_pred, zero_division=0)
# test_recall = recall_score(y_test, test_pred, zero_division=0)
# test_f1 = f1_score(y_test, test_pred, zero_division=0)

# print("\nTest results:")
# print(f"ROC AUC:    {test_roc_auc:.4f}")
# print(f"PR AUC:     {test_pr_auc:.4f}")
# print(f"Precision:  {test_precision:.4f}")
# print(f"Recall:     {test_recall:.4f}")
# print(f"F1:         {test_f1:.4f}")

# # ============================================================
# # PER-ANOMALY-TYPE BREAKDOWN (diagnostic — which scenarios NMF actually catches)
# # ============================================================
# print("\nPer-anomaly-type AUC on test set:")
# per_type_results = []
# for atype in sorted(test_df['anomaly_type'].unique()):
#     if atype == 'normal':
#         continue
#     mask = (test_df['anomaly_type'] == atype) | (test_df['is_anomaly'] == 0)
#     sub = test_df[mask]
#     X_sub_scaled = np.clip(scaler.transform(sub.drop(columns=drop_cols)), 0, None)
#     sub_recon = best_model.inverse_transform(best_model.transform(X_sub_scaled))
#     sub_scores = reconstruction_error_per_sample(X_sub_scaled, sub_recon)
#     try:
#         auc = roc_auc_score(sub['is_anomaly'], sub_scores)
#     except ValueError:
#         auc = float('nan')  # only one class present in this slice
#     n_pos = (sub['is_anomaly'] == 1).sum()
#     print(f"{atype:35s} AUC={auc:.3f}  n_anomaly={n_pos}")
#     per_type_results.append({'anomaly_type': atype, 'auc': auc, 'n_anomaly': n_pos})

# per_type_df = pd.DataFrame(per_type_results).sort_values('auc', ascending=False)
# print("\n", per_type_df.to_string(index=False))

In [ ]:
import time
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from pyod.models.iforest import IForest
from pyod.models.knn import KNN
from pyod.models.ecod import ECOD
from pyod.models.copod import COPOD

# ---------------------------------------------------------------------
# If you already loaded df, X, y, and created train/val/test splits,
# keep that part exactly as you have it.
# ---------------------------------------------------------------------

def find_best_threshold(y_true, scores):
    thresholds = np.unique(np.quantile(scores, np.linspace(0.01, 0.99, 99)))
    best = {
        "threshold": None,
        "f1": -1,
        "precision": None,
        "recall": None,
    }

    for t in thresholds:
        preds = (scores >= t).astype(int)
        p = precision_score(y_true, preds, zero_division=0)
        r = recall_score(y_true, preds, zero_division=0)
        f1 = f1_score(y_true, preds, zero_division=0)

        if f1 > best["f1"]:
            best = {
                "threshold": float(t),
                "f1": float(f1),
                "precision": float(p),
                "recall": float(r),
            }
    return best

def fit_and_score_pyod(model, model_name, X_train, X_val, y_val, X_test, y_test):
    start = time.perf_counter()

    print(f"\n{'='*70}")
    print(f"Training {model_name}")
    print(f"{'='*70}")

    model.fit(X_train)

    fit_time = time.perf_counter() - start

    # PyOD anomaly scores: higher = more anomalous
    val_scores = model.decision_function(X_val)
    test_scores = model.decision_function(X_test)

    val_roc_auc = roc_auc_score(y_val, val_scores)
    val_pr_auc = average_precision_score(y_val, val_scores)

    best = find_best_threshold(y_val, val_scores)

    test_pred = (test_scores >= best["threshold"]).astype(int)
    test_roc_auc = roc_auc_score(y_test, test_scores)
    test_pr_auc = average_precision_score(y_test, test_scores)
    test_precision = precision_score(y_test, test_pred, zero_division=0)
    test_recall = recall_score(y_test, test_pred, zero_division=0)
    test_f1 = f1_score(y_test, test_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test, test_pred).ravel()

    print(f"Finished fitting in {fit_time:.2f} seconds")
    print(f"Validation ROC-AUC={val_roc_auc:.4f} | PR-AUC={val_pr_auc:.4f}")
    print(f"Best validation threshold={best['threshold']:.6f}")
    print(f"Best validation F1={best['f1']:.4f} | Precision={best['precision']:.4f} | Recall={best['recall']:.4f}")
    print(f"Test ROC-AUC={test_roc_auc:.4f} | PR-AUC={test_pr_auc:.4f} | Precision={test_precision:.4f} | Recall={test_recall:.4f} | F1={test_f1:.4f}")
    print(f"Test confusion matrix: TP={tp}, FP={fp}, FN={fn}, TN={tn}")

    return {
        "model_name": model_name,
        "model": model,
        "fit_time_sec": round(fit_time, 2),
        "val_roc_auc": val_roc_auc,
        "val_pr_auc": val_pr_auc,
        "best_threshold": best["threshold"],
        "best_val_f1": best["f1"],
        "best_val_precision": best["precision"],
        "best_val_recall": best["recall"],
        "test_roc_auc": test_roc_auc,
        "test_pr_auc": test_pr_auc,
        "test_precision": test_precision,
        "test_recall": test_recall,
        "test_f1": test_f1,
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "test_scores": test_scores,
        "val_scores": val_scores,
    }

# ---------------------------------------------------------------------
# Models
# ---------------------------------------------------------------------
models_to_run = {
    "IForest": IForest(
        n_estimators=200,
        contamination=0.1,
        random_state=42,
    ),

    "KNN": KNN(
        n_neighbors=5,
        method="largest",
        contamination=0.1,
    ),

    "ECOD": ECOD(),

    "COPOD": COPOD(),

    "LOF": LOF(
        n_neighbors=20,
        contamination=0.1,
    ),
}

results = {}
rows = []

overall_start = time.perf_counter()

for name, model in models_to_run.items():
    out = fit_and_score_pyod(
        model=model,
        model_name=name,
        X_train=X_train_normal_scaled,
        X_val=X_val_scaled,
        y_val=y_val,
        X_test=X_test_scaled,
        y_test=y_test,
    )
    results[name] = out
    rows.append({
        "model": name,
        "fit_time_sec": out["fit_time_sec"],
        "val_roc_auc": out["val_roc_auc"],
        "val_pr_auc": out["val_pr_auc"],
        "best_val_f1": out["best_val_f1"],
        "best_val_precision": out["best_val_precision"],
        "best_val_recall": out["best_val_recall"],
        "best_threshold": out["best_threshold"],
        "test_roc_auc": out["test_roc_auc"],
        "test_pr_auc": out["test_pr_auc"],
        "test_precision": out["test_precision"],
        "test_recall": out["test_recall"],
        "test_f1": out["test_f1"],
        "tp": out["tp"],
        "fp": out["fp"],
        "fn": out["fn"],
        "tn": out["tn"],
    })

overall_time = time.perf_counter() - overall_start

results_df = pd.DataFrame(rows).sort_values(["test_pr_auc", "test_roc_auc"], ascending=False)

print("\n" + "="*80)
print("Final Comparison")
print("="*80)
print(results_df.to_string(index=False))
print(f"\nTotal runtime: {overall_time:.2f} seconds")